# 03 - Gold Dimensional Model

## Objetivo

Construir um modelo dimensional em estrela para suportar análises de negócio e responder às perguntas do case.

## Entidades

- Dimensão Marca
- Dimensão Canal
- Dimensão Região
- Dimensão Embalagem
- Dimensão Tempo

## Fato

- Fact Sales

In [0]:
# Importação das bibliotecas

from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# Leitura da camada Silver

silver_df = spark.table(
    "workspace.silver.sales_enriched"
)

print("Registros:")
print(silver_df.count())

Registros:
16151


In [0]:
# Criação da dimensão de marcas

dim_brand = (
    silver_df
    .select(
        "CE_BRAND_FLVR",
        "BRAND_NM"
    )
    .distinct()
)

display(dim_brand)

CE_BRAND_FLVR,BRAND_NM
3440,LEMON
3554,STRAWBERRY
3441,RASPBERRY
3697,GRAPE


In [0]:
# Criação da chave substituta da dimensão de marcas

from pyspark.sql.functions import monotonically_increasing_id

dim_brand = (
    dim_brand
    .withColumn(
        "brand_key",
        monotonically_increasing_id() + 1
    )
)

display(dim_brand)

CE_BRAND_FLVR,BRAND_NM,brand_key
3440,LEMON,1
3554,STRAWBERRY,2
3441,RASPBERRY,3
3697,GRAPE,4


In [0]:
# Criação da dimensão de canais

dim_channel = (
    silver_df
    .select(
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "TRADE_TYPE_DESC"
    )
    .distinct()
    .withColumn(
        "channel_key",
        monotonically_increasing_id() + 1
    )
)

display(dim_channel)

TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC,channel_key
SPORT VENUE,ENTERTAINMENT,ALCOHOLIC,1
SUPERETTE,SERVICES,MIX,2
PLANT / OFFICE,SERVICES,MIX,3
MASS MERCHANDISER,GROCERY,MIX,4
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC,5
CONVENIENCE STORE,SERVICES,MIX,6
QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC,7
SUPERMARKET,GROCERY,MIX,8
ALL OTHER,OTHER,MIX,9
OTHER EATING + DRINKING,SERVICES,MIX,10


In [0]:
# Criação da dimensão de regiões

dim_region = (
    silver_df
    .select(
        "Btlr_Org_LVL_C_Desc"
    )
    .distinct()
    .withColumn(
        "region_key",
        monotonically_increasing_id() + 1
    )
)

display(dim_region)

Btlr_Org_LVL_C_Desc,region_key
CANADA,1
NORTHEAST,2
SOUTHEAST,3
MIDWEST,4
WEST,5
SOUTHWEST,6
GREAT LAKES,7


In [0]:
# Criação da dimensão de embalagens

dim_package = (
    silver_df
    .select(
        "PKG_CAT",
        "Pkg_Cat_Desc",
        "TSR_PCKG_NM"
    )
    .distinct()
    .withColumn(
        "package_key",
        monotonically_increasing_id() + 1
    )
)

display(dim_package)

PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,package_key
N20O,20Z/600ML,.591L NRP 24L,1
N20O,20Z/600ML,20Z NRP 24L,2
N56P,500ML 6P,.5L NRP 6P,3
N20O,20Z/600ML,20z NRP 24L S,4
N20O,20Z/600ML,.591L NRP 24L *,5
N56P,500ML 6P,.5L NRP 6P S,6
N56P,500ML 6P,.5L NRP 6P*,7
N128,12Z/355M 8NR,12Z NRP 8P F,8


In [0]:
# Criação da dimensão de datas

dim_date = (
    silver_df
    .select(
        "DATE",
        "YEAR",
        "MONTH",
        "PERIOD"
    )
    .distinct()
    .withColumn(
        "date_key",
        monotonically_increasing_id() + 1
    )
)

display(dim_date)

DATE,YEAR,MONTH,PERIOD,date_key
2006-01-01,2006,1,1,1
2006-01-08,2006,1,2,2
2006-01-15,2006,1,3,3
2006-01-22,2006,1,4,4
2006-02-01,2006,2,1,5
2006-02-08,2006,2,2,6
2006-02-15,2006,2,3,7
2006-02-22,2006,2,4,8
2006-03-01,2006,3,1,9
2006-03-08,2006,3,2,10


In [0]:
print("dim_brand:", dim_brand.count())
print("dim_channel:", dim_channel.count())
print("dim_region:", dim_region.count())
print("dim_package:", dim_package.count())
print("dim_date:", dim_date.count())

dim_brand: 4
dim_channel: 31
dim_region: 7
dim_package: 8
dim_date: 13


In [0]:
# Criação da tabela fato

fact_sales = (
    silver_df.alias("s")

    .join(
        dim_brand.alias("b"),
        ["CE_BRAND_FLVR","BRAND_NM"]
    )

    .join(
        dim_channel.alias("c"),
        ["TRADE_CHNL_DESC","TRADE_GROUP_DESC","TRADE_TYPE_DESC"]
    )

    .join(
        dim_region.alias("r"),
        ["Btlr_Org_LVL_C_Desc"]
    )

    .join(
        dim_package.alias("p"),
        ["PKG_CAT","Pkg_Cat_Desc","TSR_PCKG_NM"]
    )

    .join(
        dim_date.alias("d"),
        ["DATE","YEAR","MONTH","PERIOD"]
    )

    .select(
        "date_key",
        "brand_key",
        "channel_key",
        "region_key",
        "package_key",
        "sales_volume"
    )
)

display(fact_sales.limit(10))

date_key,brand_key,channel_key,region_key,package_key,sales_volume
7,3,12,2,3,22.48
7,3,26,7,8,100.0
7,1,2,4,8,66.14
7,2,24,5,8,222.5
7,3,24,6,8,302.5
7,3,3,5,8,10.0
7,2,27,4,1,-4.22
7,1,15,2,3,59.95
7,1,6,1,4,300.0
7,1,15,4,8,85.0


In [0]:
print("Fact Sales:", fact_sales.count())

Fact Sales: 16151


In [0]:
# Gravação das dimensões

dim_brand.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_brand")

dim_channel.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_channel")

dim_region.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_region")

dim_package.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_package")

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_date")

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_sales")

print("Camada Gold criada com sucesso.")

Camada Gold criada com sucesso.
